# 02 - Exploratory data analysis

**Owner:** Yuhan Zhang (Modelling lead)  
**Depends on:** `01_data_check.ipynb`  
**Goal:** decide the final class list and document how we handle Surgery, document-type labels, duplicates, label leakage, missing text, and cross-label contradictions, so the split (Issue #2) can be frozen on clean labels.

Run top to bottom. Works in Google Colab (free tier) or locally. Only edit the CONFIG cell.

## 0. Setup

In [ ]:
import sys
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content")
elif Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

DATA_DIR = ROOT / "data"
print(f"Running in {'Colab' if IN_COLAB else 'local'}; data folder: {DATA_DIR}")

## 1. CONFIG

In [ ]:
KAGGLE_DATASET = "tboyle10/medicaltranscriptions"
CSV_PATH       = DATA_DIR / "mtsamples.csv"

TEXT_COL    = "transcription"
LABEL_COL   = "medical_specialty"

RANDOM_SEED = 42

## 2. Load the data

Same loading logic as `01_data_check.ipynb`, plus a label clean-up step (see the note below).

In [ ]:
def download_from_kaggle():
    """Colab only: download mtsamples.csv using the Kaggle key stored in Colab Secrets."""
    import os
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    get_ipython().system(f"kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_DIR} --unzip")

if not CSV_PATH.exists():
    if IN_COLAB:
        download_from_kaggle()
    else:
        raise FileNotFoundError(
            f"{CSV_PATH} not found. Download mtsamples.csv first, see data/README.md."
        )

df = pd.read_csv(CSV_PATH)

# DATA BUG: every value in medical_specialty has a leading space (e.g. " Surgery").
# Strip it once here; every downstream mapping and split depends on this.
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns")
print("Columns:", list(df.columns))
df.head()

## 3. Full label inventory

All 40 raw labels, sorted by count, with cumulative share. Basis for the long-tail threshold.

In [ ]:
counts = df[LABEL_COL].value_counts()
share = (counts / counts.sum() * 100).round(2)
cum = share.cumsum().round(2)

label_table = pd.DataFrame({"count": counts, "share_%": share, "cumulative_%": cum})
label_table.index.name = LABEL_COL
print(f"Number of raw labels: {label_table.shape[0]}")
label_table

## 4. Document-type labels vs specialties

Some labels are document types, not medical specialties. **Decision: drop them.** They do not define a routing destination, and a model that predicts a document type is not useful for the stated task.

In [ ]:
DOC_TYPE_LABELS = [
    "Consult - History and Phy.",
    "SOAP / Chart / Progress Notes",
    "Discharge Summary",
    "Emergency Room Reports",
    "Office Notes",
    "Letters",
]

print("Document-type labels to drop:")
for c in DOC_TYPE_LABELS:
    n = counts.get(c, 0)
    print(f"  {c!r}: {n} rows")
print(f"Total dropped by this rule: {counts.reindex(DOC_TYPE_LABELS).fillna(0).astype(int).sum()} rows")

## 5. Surgery overlap

Surgery is ~22% of rows. **Decision: drop the Surgery label.** 1,054 of 1,103 Surgery rows are byte-for-byte identical to a note already labelled with another specialty, so Surgery is a procedure type, not a specialty. Of the 49 unique Surgery rows, ~31 map to a real specialty (ports, breast biopsies, ERCP, etc.) and are re-labelled via the review step (Section 9); the remaining ~18 are general-surgery / wound-care procedures with no specialty home and are dropped.

In [ ]:
def normalize_text(s):
    """Lowercase, strip punctuation, collapse whitespace. Used for duplicate matching."""
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["_norm"] = df[TEXT_COL].map(normalize_text)

surgery_mask = df[LABEL_COL].eq("Surgery")
others = df.loc[~surgery_mask]
surgery_raw_dup = df.loc[surgery_mask & df[TEXT_COL].isin(set(others[TEXT_COL]))]

print(f"Surgery rows: {surgery_mask.sum()}")
print(f"Byte-for-byte identical to another specialty's note: {len(surgery_raw_dup)}")
print(f"Unique Surgery rows (re-labelled later): {surgery_mask.sum() - len(surgery_raw_dup)}")

## 6. Duplicates

Only ~4% of rows have a unique text; the rest are duplicates, most filed under 2+ labels. **Decision: keep duplicate rows but force the same normalized text into the same split** (removes leakage while keeping data, per the evaluation protocol).

In [ ]:
exact_dup = df[TEXT_COL].duplicated().sum()
norm_dup = df["_norm"].duplicated().sum()
unique_texts = (df["_norm"].value_counts() == 1).sum()

print(f"Exact duplicate rows: {exact_dup} ({exact_dup/len(df):.1%})")
print(f"Normalized duplicate rows: {norm_dup} ({norm_dup/len(df):.1%})")
print(f"Texts appearing exactly once: {unique_texts} ({unique_texts/len(df):.1%})")

## 7. Label leakage

The `keywords` and `description` columns and some note headers contain the label itself. **Decision: model on `transcription` only, drop `keywords`/`description`, and strip the first header line** of each note (it often states the specialty).

In [ ]:
def label_in_text(col):
    hit = 0
    for lab, val in zip(df[LABEL_COL], df[col]):
        if isinstance(val, str) and str(lab).lower() in val.lower():
            hit += 1
    return hit

for col in ["keywords", "description"]:
    if col in df.columns:
        n = label_in_text(col)
        print(f"{col}: label string appears in {n} rows ({n/len(df):.1%})")

def first_line(s):
    if not isinstance(s, str):
        return ""
    for line in s.splitlines():
        if line.strip():
            return line.strip()
    return ""

df["_first_line"] = df[TEXT_COL].map(first_line)
n = label_in_text("_first_line")
print(f"first header line: label string appears in {n} rows ({n/len(df):.1%})")

## 8. Missing or empty text

**Decision: drop rows with missing transcription** (33 rows, 0.7%).

In [ ]:
print(f"Missing transcriptions: {df[TEXT_COL].isna().sum()}")
print(f"Empty transcriptions:   {(df[TEXT_COL].astype(str).str.strip() == '').sum()}")
print(f"Missing labels:         {df[LABEL_COL].isna().sum()}")

## 9. Cross-label contradictions

After dropping document types and Surgery, the same note text is still filed under 2+ *real* specialties (e.g. an angiogram under both Cardiovascular and Radiology). A single-label task requires one label per text.

**Decision: assign one primary specialty per text via an LLM first pass + human review** (table `data/review_pending_labels.csv`), rather than dropping the rows. This preserves ~2,300 usable texts and is disclosed in the Use of GenAI log.

In [ ]:
DOC_TYPES = DOC_TYPE_LABELS  # defined in Section 4
clean = df[df[TEXT_COL].notna() & ~df[LABEL_COL].isin(DOC_TYPES) & df[LABEL_COL].ne("Surgery")].copy()

g = clean.groupby("_norm").agg(cnt=("sample_name", "size"), nlab=(LABEL_COL, "nunique"))
unique = (g["cnt"] == 1).sum()
cross = ((g["cnt"] > 1) & (g["nlab"] > 1)).sum()

print(f"After cleaning: {len(clean)} rows")
print(f"Unambiguous unique texts: {unique}")
print(f"Cross-label contradictory texts: {cross} (resolved via review table)")

## 10. Final class list (draft, pending team confirmation)

Ten classes, mapping raw labels into specialty families. Rare labels with no natural home are dropped.

| Final class | Raw labels mapped in |
|---|---|
| Cardiovascular / Pulmonary | Cardiovascular / Pulmonary |
| Orthopedic | Orthopedic, Podiatry |
| Radiology | Radiology |
| General Medicine | General Medicine, Pain Management, Physical Medicine - Rehab, Sleep Medicine, Endocrinology, Bariatrics, Rheumatology, Diets and Nutritions, Allergy / Immunology |
| Gastroenterology | Gastroenterology |
| Neurology | Neurology, Neurosurgery |
| Urology / Nephrology | Urology, Nephrology |
| Obstetrics / Gynecology / Pediatrics | Obstetrics / Gynecology, Pediatrics - Neonatal |
| ENT / Ophthalmology | ENT - Otolaryngology, Ophthalmology, Dentistry, Cosmetic / Plastic Surgery |
| Hematology - Oncology | Hematology - Oncology, Lab Medicine - Pathology |

Dropped (rare, no natural home, or no specialty mapping): Dermatology, IME-QME-Work Comp etc., Chiropractic, Speech - Language, Autopsy, Hospice - Palliative Care, Psychiatry / Psychology (still to confirm), and ~18 general-surgery / wound-care procedures from the unique Surgery rows (AI + human review deemed them droppable).

## 11. Decisions checklist (Issue #1)

- [x] Strip leading spaces from labels
- [x] Drop document-type labels (6 labels)
- [x] Drop the Surgery label (keep subspecialty copies; re-label ~31 unique rows, drop ~18 general-surgery rows)
- [x] Duplicates: keep rows, force same text into the same split
- [x] Leakage: model on transcription only, drop keywords/description, strip header line
- [x] Missing text: drop 33 rows
- [x] Cross-label contradictions: LLM first pass + human review
- [ ] Final class list: confirm with team (Section 10)

Clear all outputs before committing (raw note text must not enter the repo).